# Unsupervised Learning — the Math, Worked Out in Code

This notebook is the hands-on companion to the *Unsupervised Learning* slide deck.
Every algorithm below is implemented **twice**: once **from scratch** with plain NumPy
(so you can see every term of the equation actually being computed), and once with the
matching **scikit-learn / SciPy** call (so you can trust the from-scratch version is correct).

Each section follows the same rhythm as the deck's side panel:
- **Theory** — what the method is and when to reach for it
- **Math** — the equation, computed line by line
- **Practice** — a small, real calculation you can change and re-run

> Richard Feynman's test: if you can't code the equation from scratch, you don't
> understand it yet. That's the whole point of this notebook.


## 0 · Setup

Run this first. It sets your Groq API key (used later for an optional `ask_ai()` helper),
and imports everything the rest of the notebook needs.


In [ ]:
import os
from getpass import getpass

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "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"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Setup complete. GROQ_API_KEY is set:", bool(os.environ.get("GROQ_API_KEY")))


### Optional: `ask_ai()` — a tutor in a function

The API key above is a **placeholder** — swap it for your own free key from
[console.groq.com](https://console.groq.com) to make this actually respond.
Until then, it fails gracefully and tells you why.

Use it anywhere in this notebook, e.g. `ask_ai("Why do we square the distance in K-Means?")`.


In [ ]:
import requests

def ask_ai(question, model="llama-3.1-8b-instant", temperature=0.3):
    # Ask a Groq-hosted LLM to explain a concept, Feynman-style.
    # Requires a real GROQ_API_KEY (get one free at https://console.groq.com).
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {os.environ['GROQ_API_KEY']}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": (
                "You are a friendly math tutor in the style of Richard Feynman. "
                "Explain the concept simply, in plain English, then give one tiny "
                "worked numeric example. Keep it under 150 words."
            )},
            {"role": "user", "content": question},
        ],
        "temperature": temperature,
    }
    try:
        r = requests.post(url, headers=headers, json=payload, timeout=20)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"]
    except Exception as e:
        return (
            "[ask_ai couldn't reach Groq — this notebook ships with a placeholder key.\n"
            f" Swap in a real key from https://console.groq.com to use this. ({type(e).__name__})]"
        )

print(ask_ai("In one sentence, why do we square the distance in the K-Means objective?"))


---
## 1 · Distance Metrics

**Theory.** Every distance-based algorithm in this notebook (K-Means, DBSCAN, Hierarchical,
LOF...) needs one thing first: a number that answers *"how similar are these two points?"*
That number is a **distance metric**, and the metric you pick quietly decides what
"similar" even means.

**Math.**
$$d_{Euclidean}(x,y)=\sqrt{\sum_i (x_i-y_i)^2} \qquad
d_{Manhattan}(x,y)=\sum_i |x_i-y_i| \qquad
d_{Cosine}(x,y)=1-\frac{x\cdot y}{\|x\|\,\|y\|}$$

- **Euclidean** — straight-line "as the crow flies" distance.
- **Manhattan** — sum of per-axis steps, like walking city blocks.
- **Cosine** — the *angle* between two vectors; ignores magnitude entirely (great for text).


In [ ]:
def euclidean(x, y):
    x, y = np.array(x, dtype=float), np.array(y, dtype=float)
    return np.sqrt(np.sum((x - y) ** 2))

def manhattan(x, y):
    x, y = np.array(x, dtype=float), np.array(y, dtype=float)
    return np.sum(np.abs(x - y))

def cosine_distance(x, y):
    x, y = np.array(x, dtype=float), np.array(y, dtype=float)
    return 1 - (x @ y) / (np.linalg.norm(x) * np.linalg.norm(y))

x, y = (1, 2), (4, 6)
print(f"Euclidean(x, y) = {euclidean(x, y):.3f}")
print(f"Manhattan(x, y) = {manhattan(x, y):.3f}")
print(f"Cosine distance(x, y) = {cosine_distance(x, y):.4f}")

# --- sanity check against SciPy's implementations ---
from scipy.spatial.distance import euclidean as sp_euclidean, cityblock as sp_manhattan, cosine as sp_cosine
assert np.isclose(euclidean(x, y), sp_euclidean(x, y))
assert np.isclose(manhattan(x, y), sp_manhattan(x, y))
assert np.isclose(cosine_distance(x, y), sp_cosine(x, y))
print("\nAll three match SciPy's implementations exactly ✓")


In [ ]:
plt.figure()
plt.scatter(*x, s=90, zorder=3, label="x = (1, 2)")
plt.scatter(*y, s=90, zorder=3, label="y = (4, 6)")
plt.plot([x[0], y[0]], [x[1], y[1]], "b--", label=f"Euclidean = {euclidean(x, y):.2f}")
plt.plot([x[0], y[0], y[0]], [x[1], x[1], y[1]], color="orange", linestyle=":",
         label=f"Manhattan = {manhattan(x, y):.2f}")
plt.legend(); plt.axis("equal")
plt.title("Euclidean (straight line) vs. Manhattan (grid steps)")
plt.show()


---
## 2 · K-Means

**Theory.** K-Means splits data into $K$ groups by repeating two steps: assign every point
to its nearest centroid, then move each centroid to the mean of its assigned points
(*Lloyd's algorithm*). It always converges — but not always to the *best* answer, only a
*locally* best one.

**Math.**
$$J=\sum_{k}\sum_{x\in C_k}\|x-\mu_k\|^2$$

$J$ (the **inertia**) can only decrease every iteration. We implement exactly this loop below.


In [ ]:
def kmeans_from_scratch(X, k, n_iter=20, seed=0):
    rng = np.random.default_rng(seed)
    centroids = X[rng.choice(len(X), k, replace=False)].astype(float)
    history = {"inertia": []}
    for _ in range(n_iter):
        dists = cdist(X, centroids)                 # every point to every centroid
        labels = np.argmin(dists, axis=1)            # step 1: assign
        inertia = sum(np.sum((X[labels == j] - centroids[j]) ** 2) for j in range(k))
        history["inertia"].append(inertia)
        new_centroids = np.array([
            X[labels == j].mean(axis=0) if np.any(labels == j) else centroids[j]
            for j in range(k)
        ])                                             # step 2: update
        if np.allclose(new_centroids, centroids):
            break
        centroids = new_centroids
    return labels, centroids, history

from sklearn.datasets import make_blobs
X, y_true = make_blobs(n_samples=300, centers=3, cluster_std=0.9, random_state=42)

labels, centroids, hist = kmeans_from_scratch(X, k=3, seed=0)
print("J at every iteration:", [round(v, 1) for v in hist["inertia"]])


### Why did it stall so high? — initialization matters

If the J above looks stuck around 5500 instead of dropping close to ~460, that's not a bug —
it's the exact "Watch out for" from the Theory panel: **K-Means is sensitive to its random
starting centroids** and can get trapped in a bad local optimum. Let's prove it by trying
several random seeds and keeping the best one — exactly what scikit-learn's `n_init=10`
does automatically.


In [ ]:
best = None
for seed in range(10):
    lab, cen, h = kmeans_from_scratch(X, k=3, seed=seed)
    final_J = h["inertia"][-1]
    print(f"seed={seed}: final J = {final_J:8.1f}")
    if best is None or final_J < best[2]:
        best = (lab, cen, final_J)

labels, centroids, best_J = best
print(f"\nBest of 10 random restarts: J = {best_J:.1f}")

from sklearn.cluster import KMeans
skl = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
print(f"scikit-learn (n_init=10) inertia: {skl.inertia_:.1f}  <- matches our best restart")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=20, alpha=.8)
axes[0].scatter(centroids[:, 0], centroids[:, 1], c="black", marker="X", s=150)
axes[0].set_title("K-Means from scratch — best of 10 restarts")

axes[1].plot(range(10), [kmeans_from_scratch(X, 3, seed=s)[2]["inertia"][-1] for s in range(10)], "o-")
axes[1].set_xlabel("random seed"); axes[1].set_ylabel("final J")
axes[1].set_title("Same algorithm, 10 different starting points")
plt.tight_layout(); plt.show()


---
## 3 · Choosing K — Elbow Method & Silhouette Score

**Theory.** K-Means can't tell you the "right" $K$ — you find it with diagnostics computed
*after* clustering at several candidate values of $K$.

**Math.**
$$\text{elbow: look for the bend in } WCSS(K) \qquad\qquad
s(i)=\frac{b(i)-a(i)}{\max(a(i),b(i))}$$

- $a(i)$ — average distance from point $i$ to everyone else **in its own cluster** (cohesion).
- $b(i)$ — average distance from point $i$ to the **nearest other cluster** (separation).
- Averaging $s(i)$ over every point gives one overall quality score for a clustering, from $-1$ to $1$.


In [ ]:
def silhouette_from_scratch(X, labels):
    n = len(X)
    D = cdist(X, X)
    s = np.zeros(n)
    unique_labels = np.unique(labels)
    for i in range(n):
        own = labels[i]
        in_cluster = labels == own
        if in_cluster.sum() <= 1:
            s[i] = 0.0
            continue
        a_i = D[i, in_cluster].sum() / (in_cluster.sum() - 1)   # exclude the point itself
        b_i = min(D[i, labels == other].mean() for other in unique_labels if other != own)
        s[i] = (b_i - a_i) / max(a_i, b_i)
    return s

from sklearn.metrics import silhouette_score

wcss, sil = [], []
Ks = range(2, 9)
for k in Ks:
    lab, cen, h = kmeans_from_scratch(X, k, n_iter=30, seed=2)   # a seed that behaves well
    wcss.append(h["inertia"][-1])
    s_manual = silhouette_from_scratch(X, lab)
    sil.append(s_manual.mean())
    assert np.isclose(s_manual.mean(), silhouette_score(X, lab), atol=1e-6)

print("Our silhouette scores match scikit-learn's silhouette_score() exactly ✓")
print("Best K by silhouette:", list(Ks)[int(np.argmax(sil))])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(list(Ks), wcss, "o-"); axes[0].set_title("Elbow method"); axes[0].set_xlabel("K"); axes[0].set_ylabel("WCSS (J)")
axes[1].plot(list(Ks), sil, "o-", color="darkorange"); axes[1].set_title("Silhouette score"); axes[1].set_xlabel("K"); axes[1].set_ylabel("avg s(i)")
plt.tight_layout(); plt.show()


---
## 4 · The Gaussian Formula & Gaussian Mixture Models

**Theory.** A GMM assumes the data was generated by a handful of overlapping bell curves,
and recovers each one's parameters via **Expectation-Maximization (EM)**: guess soft
cluster memberships (E-step), then update each Gaussian to fit those memberships (M-step).

**Math.**
$$f(x)=\frac{1}{\sigma\sqrt{2\pi}}\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)
\qquad\qquad
\gamma_{ik}=\frac{\pi_k\,\mathcal N(x_i\mid\mu_k,\Sigma_k)}{\sum_j \pi_j\,\mathcal N(x_i\mid\mu_j,\Sigma_j)}$$

$\gamma_{ik}$ ("responsibility") is exactly *what fraction of point $i$'s probability mass
came from cluster $k$* — that fraction becomes its soft membership weight.


In [ ]:
def gaussian_pdf(x, mu, sigma):
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

xs = np.linspace(-5, 5, 200)
plt.figure()
for sigma, style in [(0.5, "-"), (1.0, "--"), (2.0, ":")]:
    plt.plot(xs, gaussian_pdf(xs, 0, sigma), style, label=f"σ = {sigma}")
plt.legend(); plt.title(r"$f(x)=\frac{1}{\sigma\sqrt{2\pi}}\exp(-(x-\mu)^2/2\sigma^2)$")
plt.show()


In [ ]:
def multivariate_gaussian_pdf(X, mu, cov):
    d = len(mu)
    diff = X - mu
    inv, det = np.linalg.inv(cov), np.linalg.det(cov)
    norm_const = 1 / np.sqrt((2 * np.pi) ** d * det)
    exponent = -0.5 * np.einsum("ij,jk,ik->i", diff, inv, diff)
    return norm_const * np.exp(exponent)

def gmm_em_from_scratch(X, k, n_iter=30, seed=0):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    mu = X[rng.choice(n, k, replace=False)].astype(float)
    cov = [np.cov(X.T) + 1e-6 * np.eye(d) for _ in range(k)]
    pi = np.ones(k) / k
    ll_history = []
    for _ in range(n_iter):
        # --- E-step: compute responsibilities gamma_ik ---
        resp = np.column_stack([pi[j] * multivariate_gaussian_pdf(X, mu[j], cov[j]) for j in range(k)])
        total = resp.sum(axis=1, keepdims=True)
        ll_history.append(np.sum(np.log(total)))
        resp = resp / total
        # --- M-step: update each Gaussian using its responsibilities ---
        Nk = resp.sum(axis=0)
        for j in range(k):
            mu[j] = (resp[:, j:j+1] * X).sum(axis=0) / Nk[j]
            diff = X - mu[j]
            cov[j] = (resp[:, j:j+1] * diff).T @ diff / Nk[j] + 1e-6 * np.eye(d)
            pi[j] = Nk[j] / n
    return mu, cov, pi, resp, ll_history

mu, cov, pi, resp, ll = gmm_em_from_scratch(X, k=3, n_iter=30)
labels_gmm = resp.argmax(axis=1)

from sklearn.mixture import GaussianMixture
skl_gmm = GaussianMixture(n_components=3, random_state=42).fit(X)
print(f"Our average log-likelihood:      {ll[-1] / len(X):.4f}")
print(f"scikit-learn average log-likelihood: {skl_gmm.score(X):.4f}   <- should be close")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(ll, "o-"); axes[0].set_title("Log-likelihood rises every EM iteration")
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("total log-likelihood")

axes[1].scatter(X[:, 0], X[:, 1], c=labels_gmm, cmap="tab10", s=20, alpha=.8)
axes[1].scatter(mu[:, 0], mu[:, 1], c="black", marker="X", s=150)
axes[1].set_title("GMM — hard label shown, but membership γ is soft")
plt.tight_layout(); plt.show()


---
## 5 · Principal Component Analysis (PCA)

**Theory.** PCA finds the directions the data varies most along, and re-expresses the data
using those directions as new axes — the ones that barely vary get dropped.

**Math.**
$$\text{maximize } \operatorname{Var}(w\cdot X)\ \text{ s.t. } \|w\|=1
\qquad\Longleftrightarrow\qquad
\operatorname{Cov}(X)\,v=\lambda v$$

The eigenvectors $v$ of the covariance matrix are the directions that are only *stretched*,
never rotated — exactly the "pure" directions of variance PCA is looking for. The eigenvalue
$\lambda$ is how much variance lies along that direction.


In [ ]:
def pca_from_scratch(X, n_components=2):
    Xc = X - X.mean(axis=0)
    cov = np.cov(Xc.T)
    eigvals, eigvecs = np.linalg.eigh(cov)          # eigh: cov is symmetric
    order = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    explained_ratio = eigvals / eigvals.sum()
    components = eigvecs[:, :n_components]
    X_proj = Xc @ components
    return X_proj, components, eigvals, explained_ratio

rng = np.random.default_rng(0)
Z = rng.normal(size=(300, 2)) @ np.array([[2.5, 1.2], [0, 0.6]])   # a correlated, elongated cloud

X_proj, comps, eigvals, ratio = pca_from_scratch(Z, n_components=2)
print("Explained variance ratio (λ_k / Σλ):", np.round(ratio, 3))

# --- cross-check: SVD of the centered data gives the same eigenvalues ---
Zc = Z - Z.mean(axis=0)
U, S, Vt = np.linalg.svd(Zc, full_matrices=False)
print("Eigenvalues via Cov(X):     ", np.round(eigvals, 3))
print("Eigenvalues via SVD (S²/n-1):", np.round((S ** 2) / (len(Z) - 1), 3))

from sklearn.decomposition import PCA
skl_pca = PCA(n_components=2).fit(Z)
print("scikit-learn explained_variance_ratio_:", np.round(skl_pca.explained_variance_ratio_, 3))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(Z[:, 0], Z[:, 1], s=15, alpha=.6)
origin = Z.mean(axis=0)
for i, col in enumerate(["crimson", "darkorange"]):
    vec = comps[:, i] * np.sqrt(eigvals[i]) * 2
    ax[0].arrow(*origin, *vec, width=0.03, color=col, length_includes_head=True, label=f"PC{i+1}")
ax[0].set_title("Data + principal axes (PC1 = direction of max variance)")
ax[0].axis("equal"); ax[0].legend()

ax[1].bar(["PC1", "PC2"], ratio, color=["crimson", "darkorange"])
ax[1].set_title("Explained variance ratio (scree)")
plt.tight_layout(); plt.show()


---
## 6 · SVD — Rotate, Stretch, Rotate

**Theory.** Any matrix, however complicated, factors into exactly three simple geometric
moves. This single decomposition underlies PCA, image compression, recommender systems,
and robust least-squares solving.

**Math.**
$$X=U\Sigma V^{\mathsf T}$$

$V^{\mathsf T}$ rotates the input to align with the transformation's natural axes, $\Sigma$
stretches along those axes by the *singular values*, and $U$ rotates the result into its
final orientation.

We'll also verify the **Eckart–Young theorem**: truncating the SVD to rank $k$ gives the
*provably best possible* rank-$k$ approximation of a matrix — nothing else can beat it.


In [ ]:
A = np.array([[3, 1], [1, 2]], dtype=float)
U, S, Vt = np.linalg.svd(A)
print("A =\n", A)
print("\nU =\n", np.round(U, 3))
print("\nΣ  =", np.round(S, 3))
print("\nVᵀ =\n", np.round(Vt, 3))
print("\nU Σ Vᵀ reconstructs A exactly:", np.allclose(U @ np.diag(S) @ Vt, A))


In [ ]:
# Eckart-Young: no rank-k matrix can approximate M better than truncated SVD does
rng = np.random.default_rng(1)
M = rng.normal(size=(20, 20))
U, S, Vt = np.linalg.svd(M, full_matrices=False)

k = 5
M_k = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
error_svd = np.linalg.norm(M - M_k, "fro")

best_random_error = np.inf
for _ in range(50):
    R = rng.normal(size=(20, k)) @ rng.normal(size=(k, 20))
    best_random_error = min(best_random_error, np.linalg.norm(M - R, "fro"))

print(f"Truncated-SVD rank-{k} error:        {error_svd:.3f}")
print(f"Best of 50 random rank-{k} attempts:  {best_random_error:.3f}")
print("SVD wins:", error_svd <= best_random_error, " — it is mathematically guaranteed to.")

plt.figure(); plt.plot(S, "o-")
plt.title("Singular values — the 'stretch' amount along each axis")
plt.xlabel("component index"); plt.ylabel("singular value"); plt.show()


---
## 7 · DBSCAN

**Theory.** DBSCAN defines clusters as dense regions separated by sparse ones — not by
distance to a center. That means it can find clusters of *any shape*, unlike K-Means.

**Math.**
$$N_\varepsilon(x)=\{y: \operatorname{dist}(x,y)\le\varepsilon\}
\qquad\qquad
x \text{ is a } \textbf{core point} \text{ if } |N_\varepsilon(x)|\ge \text{minPts}$$

A cluster is the union of every point reachable through a chain of overlapping
$\varepsilon$-neighborhoods anchored by at least one core point.


In [ ]:
def dbscan_from_scratch(X, eps, min_pts):
    n = len(X)
    D = cdist(X, X)
    neighbors = [np.where(D[i] <= eps)[0] for i in range(n)]
    core = np.array([len(neighbors[i]) >= min_pts for i in range(n)])
    labels = np.full(n, -1)          # -1 means "noise" until proven otherwise
    visited = np.zeros(n, dtype=bool)
    cluster_id = 0
    for i in range(n):
        if visited[i] or not core[i]:
            continue
        visited[i] = True
        labels[i] = cluster_id
        queue = list(neighbors[i])
        while queue:                  # flood-fill outward through core points
            j = queue.pop()
            if not visited[j]:
                visited[j] = True
                labels[j] = cluster_id
                if core[j]:
                    queue.extend(p for p in neighbors[j] if not visited[p])
            elif labels[j] == -1:
                labels[j] = cluster_id
        cluster_id += 1
    return labels, core

from sklearn.datasets import make_moons
Xm, _ = make_moons(n_samples=300, noise=0.06, random_state=42)
labels_db, core_mask = dbscan_from_scratch(Xm, eps=0.2, min_pts=5)

n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
print(f"Clusters found: {n_clusters}   Noise points: {np.sum(labels_db == -1)}")

from sklearn.cluster import DBSCAN
skl_db = DBSCAN(eps=0.2, min_samples=5).fit(Xm)
agreement = np.mean((labels_db == -1) == (skl_db.labels_ == -1))
print(f"Agreement with scikit-learn on which points are noise: {agreement*100:.1f}%")


In [ ]:
plt.figure()
plt.scatter(Xm[:, 0], Xm[:, 1], c=labels_db, cmap="tab10", s=20)
plt.title("DBSCAN from scratch on 'two moons' — K-Means would fail here")
plt.show()


---
## 8 · Hierarchical Clustering & Linkage

**Theory.** Agglomerative clustering starts with every point as its own cluster and
repeatedly merges the two closest ones. The definition of "closest cluster" (the
**linkage**) is the entire objective function — and changing it changes the shape of
clusters you get.

**Math.**

$$\text{single: } \min\ d(a,b) \qquad
\text{complete: } \max\ d(a,b) \qquad
\text{Ward: } \min\ \text{(increase in within-cluster variance)}$$


In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

Xh, _ = make_blobs(n_samples=20, centers=3, cluster_std=0.6, random_state=7)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, method in zip(axes, ["single", "complete", "ward"]):
    Z = linkage(Xh, method=method)
    dendrogram(Z, ax=ax, no_labels=True)
    ax.set_title(f"linkage = '{method}'")
plt.tight_layout(); plt.show()

# cut the Ward dendrogram at K=3
Z = linkage(Xh, method="ward")
clusters = fcluster(Z, t=3, criterion="maxclust")

plt.figure()
plt.scatter(Xh[:, 0], Xh[:, 1], c=clusters, cmap="tab10", s=70)
plt.title("Agglomerative clustering (Ward linkage), cut at K=3")
plt.show()


---
## 9 · Mahalanobis Distance & Elliptic Envelope

**Theory.** Ordinary distance treats every direction as equally likely. Mahalanobis
distance rescales each direction by how much the data *actually* varies along it — so
"far" in a rarely-varying direction counts for more than "far" in a commonly-varying one.

**Math.**
$$D(x)=\sqrt{(x-\mu)^{\mathsf T}\Sigma^{-1}(x-\mu)}
\qquad\qquad
\text{flag if } D(x)^2 > \chi^2_{d,\,1-\alpha}$$

Since $D(x)^2$ follows a known chi-squared distribution when the data is Gaussian, we get a
statistically principled cutoff — not an arbitrary threshold — for "too far."


In [ ]:
def mahalanobis(x, mu, cov_inv):
    diff = x - mu
    return np.sqrt(diff @ cov_inv @ diff)

rng = np.random.default_rng(2)
normal_pts = rng.multivariate_normal([0, 0], [[4, 3], [3, 4]], size=200)
outliers = np.array([[5, -5], [-6, 6]])
all_pts = np.vstack([normal_pts, outliers])

mu = normal_pts.mean(axis=0)
cov_inv = np.linalg.inv(np.cov(normal_pts.T))
D_mahal = np.array([mahalanobis(p, mu, cov_inv) for p in all_pts])

from scipy.stats import chi2
threshold = np.sqrt(chi2.ppf(0.975, df=2))          # 97.5% cutoff, 2 degrees of freedom
flagged = D_mahal > threshold
print(f"Chi-squared threshold (df=2, 97.5%): {threshold:.2f}")
print(f"Flagged as anomalies: {flagged.sum()} points")

from sklearn.covariance import EllipticEnvelope
ee = EllipticEnvelope(contamination=0.02, random_state=0).fit(normal_pts)
pred = ee.predict(all_pts)                            # -1 = outlier
print(f"scikit-learn EllipticEnvelope flags: {(pred == -1).sum()} points")


In [ ]:
plt.figure()
plt.scatter(all_pts[~flagged, 0], all_pts[~flagged, 1], label="normal", s=25)
plt.scatter(all_pts[flagged, 0], all_pts[flagged, 1], color="crimson", label="flagged", s=70)
plt.legend(); plt.title("Mahalanobis distance thresholding")
plt.show()


---
## 10 · Kernel Density Estimation (KDE)

**Theory.** KDE estimates a probability density by placing a small bell-shaped "bump" on
every data point, then summing all the bumps into one smooth curve. No assumption is made
about how many "clusters" the true density has — it's fully nonparametric.

**Math.**
$$\hat f(x)=\frac{1}{nh}\sum_i K\!\left(\frac{x-x_i}{h}\right)$$

The **bandwidth** $h$ matters far more than the kernel shape: too small gives a spiky,
overfit density; too large blurs everything into one blob.


In [ ]:
def kde_from_scratch(x_eval, data, bandwidth):
    data = np.asarray(data)
    diffs = (x_eval[:, None] - data[None, :]) / bandwidth
    kernels = np.exp(-0.5 * diffs ** 2) / np.sqrt(2 * np.pi)
    return kernels.sum(axis=1) / (len(data) * bandwidth)

rng = np.random.default_rng(3)
data = np.concatenate([rng.normal(-2, 0.5, 40), rng.normal(2, 0.8, 60)])
x_eval = np.linspace(-6, 6, 400)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, h in zip(axes, [0.1, 0.4, 1.5]):
    ax.plot(x_eval, kde_from_scratch(x_eval, data, h))
    ax.scatter(data, np.zeros_like(data), marker="|", color="k", alpha=.5)
    ax.set_title(f"bandwidth h = {h}")
plt.suptitle("Too small = spiky. Too large = blurry.")
plt.tight_layout(); plt.show()

# cross-check against scikit-learn's KernelDensity (same formula, same bandwidth units)
from sklearn.neighbors import KernelDensity
skl_kde = KernelDensity(kernel="gaussian", bandwidth=0.4).fit(data.reshape(-1, 1))
density_skl = np.exp(skl_kde.score_samples(x_eval.reshape(-1, 1)))
density_ours = kde_from_scratch(x_eval, data, 0.4)
print("Max difference from scikit-learn's KernelDensity:", np.max(np.abs(density_skl - density_ours)))


---
## 11 · Association Rules — Apriori

**Theory.** Apriori finds items that frequently co-occur, growing candidate itemsets one
item at a time and pruning aggressively using one simple rule: *if an itemset is frequent,
every subset of it must be frequent too.*

**Math.**
$$\text{support}(A)=\frac{\text{count}(A)}{N}
\qquad
\text{confidence}(A\!\to\!B)=\frac{\text{support}(A\cup B)}{\text{support}(A)}
\qquad
\text{lift}(A\!\to\!B)=\frac{\text{confidence}(A\!\to\!B)}{\text{support}(B)}$$

Lift $>1$ means $A$ and $B$ genuinely co-occur more than random chance predicts —
confidence alone can't tell you that.


In [ ]:
transactions = [
    {"bread", "milk"},
    {"bread", "butter", "milk"},
    {"beer", "diaper"},
    {"bread", "milk", "butter"},
    {"beer", "diaper", "chips"},
]

def support(itemset, transactions):
    itemset = set(itemset)
    return sum(1 for t in transactions if itemset.issubset(t)) / len(transactions)

def confidence(A, B, transactions):
    return support(A | B, transactions) / support(A, transactions)

def lift(A, B, transactions):
    return confidence(A, B, transactions) / support(B, transactions)

A, B = {"bread"}, {"milk"}
print(f"support(bread)            = {support(A, transactions):.2f}")
print(f"confidence(bread -> milk) = {confidence(A, B, transactions):.2f}")
print(f"lift(bread -> milk)       = {lift(A, B, transactions):.2f}   (>1 -> genuinely associated)")

A2, B2 = {"bread"}, {"beer"}
print(f"lift(bread -> beer)       = {lift(A2, B2, transactions):.2f}   (0 -> never co-occur)")


In [ ]:
from itertools import combinations

def apriori(transactions, min_support=0.3):
    # Frequent-itemset mining using the Apriori pruning property.
    items = sorted(set().union(*transactions))
    freq_itemsets = {}
    k, candidates = 1, [frozenset([i]) for i in items]
    while candidates:
        counts = {c: support(c, transactions) for c in candidates}
        survivors = {c: s for c, s in counts.items() if s >= min_support}
        if not survivors:
            break
        freq_itemsets.update(survivors)
        prev_items = list(survivors.keys())          # only extend itemsets that survived
        next_candidates = {a | b for a, b in combinations(prev_items, 2) if len(a | b) == k + 1}
        candidates = list(next_candidates)
        k += 1
    return freq_itemsets

freq = apriori(transactions, min_support=0.4)
for itemset, s in sorted(freq.items(), key=lambda kv: -kv[1]):
    print(f"{set(itemset)}: support = {s:.2f}")


---
## 12 · Isolation Forest

**Theory.** Instead of modeling what "normal" looks like, Isolation Forest measures how
*easy* each point is to isolate using random cuts. Anomalies isolate almost immediately;
normal points, surrounded by neighbors, take many more cuts to wall off alone.

**Math.**
$$s(x,n)=2^{-E(h(x))/c(n)}
\qquad\qquad
c(n)=2H(n-1)-\frac{2(n-1)}{n}$$

$E(h(x))$ is the average path length (number of splits) to isolate $x$ across many random
trees; $c(n)$ is the expected path length for a *normal* point, used to normalize the score
into a clean 0–1 range (near 1 = anomaly, near 0.5 = normal).


In [ ]:
class IsoNode:
    def __init__(self, depth):
        self.depth = depth
        self.left = self.right = None
        self.split_feat = self.split_val = None
        self.size = 0

def c_factor(n):
    # Expected path length for n points in a random binary tree.
    if n <= 1:
        return 0
    return 2 * (np.log(n - 1) + 0.5772156649) - 2 * (n - 1) / n   # harmonic-number approx.

def build_itree(X, depth, max_depth, rng):
    node = IsoNode(depth); node.size = len(X)
    if depth >= max_depth or len(X) <= 1:
        return node
    feat = rng.integers(0, X.shape[1])
    lo, hi = X[:, feat].min(), X[:, feat].max()
    if lo == hi:
        return node
    split = rng.uniform(lo, hi)
    node.split_feat, node.split_val = feat, split
    left_mask = X[:, feat] < split
    node.left = build_itree(X[left_mask], depth + 1, max_depth, rng)
    node.right = build_itree(X[~left_mask], depth + 1, max_depth, rng)
    return node

def path_length(x, node):
    if node.split_feat is None:
        return node.depth + c_factor(node.size)
    branch = node.left if x[node.split_feat] < node.split_val else node.right
    return path_length(x, branch)

def isolation_forest_from_scratch(X, n_trees=100, sample_size=64, seed=0):
    rng = np.random.default_rng(seed)
    max_depth = int(np.ceil(np.log2(max(sample_size, 2))))
    trees = [
        build_itree(X[rng.choice(len(X), min(sample_size, len(X)), replace=False)], 0, max_depth, rng)
        for _ in range(n_trees)
    ]
    def score(x):
        avg_path = np.mean([path_length(x, t) for t in trees])
        return 2 ** (-avg_path / c_factor(sample_size))
    return score

rng = np.random.default_rng(4)
Xn = np.vstack([rng.normal(0, 1, size=(200, 2)), [[6, 6]]])   # one planted outlier
score_fn = isolation_forest_from_scratch(Xn, n_trees=100, sample_size=64, seed=1)
scores = np.array([score_fn(x) for x in Xn])

print(f"Anomaly score of the planted outlier (6,6): {scores[-1]:.3f}  (near 1 = anomalous)")
print(f"Median score of the normal points:          {np.median(scores[:-1]):.3f}  (near 0.5 = normal)")

from sklearn.ensemble import IsolationForest
skl_if = IsolationForest(random_state=0).fit(Xn)
outlier_is_most_anomalous = skl_if.decision_function(Xn)[-1] < skl_if.decision_function(Xn)[:-1].min()
print("scikit-learn agrees the planted point is the most anomalous:", outlier_is_most_anomalous)


In [ ]:
plt.figure()
sc = plt.scatter(Xn[:, 0], Xn[:, 1], c=scores, cmap="coolwarm", s=25)
plt.colorbar(sc, label="anomaly score s(x,n)")
plt.title("Isolation Forest — fewer cuts to isolate = more anomalous")
plt.show()


---
## Recap

| Concept | Core equation | What it buys you |
|---|---|---|
| Distance metrics | $\sqrt{\sum(x_i-y_i)^2}$ | A definition of "similar" |
| K-Means | $J=\sum_k\sum_{x\in C_k}\|x-\mu_k\|^2$ | Fast, simple partitioning |
| Silhouette | $s(i)=\frac{b(i)-a(i)}{\max(a(i),b(i))}$ | A way to pick K |
| Gaussian / GMM | $\gamma_{ik}=\pi_k\mathcal N(x_i\mid\mu_k,\Sigma_k)/\cdots$ | Soft, elliptical clusters |
| PCA | $\operatorname{Cov}(X)v=\lambda v$ | Fewer, uncorrelated axes |
| SVD | $X=U\Sigma V^{\mathsf T}$ | The best possible low-rank compression |
| DBSCAN | $\lvert N_\varepsilon(x)\rvert \ge \text{minPts}$ | Arbitrary-shaped clusters |
| Mahalanobis | $D(x)=\sqrt{(x-\mu)^{\mathsf T}\Sigma^{-1}(x-\mu)}$ | Distance that respects correlation |
| KDE | $\hat f(x)=\frac{1}{nh}\sum_i K(\frac{x-x_i}{h})$ | A smooth density from raw points |
| Apriori | $\text{lift}=\text{confidence}/\text{support}(B)$ | Rules that beat random chance |
| Isolation Forest | $s(x,n)=2^{-E(h(x))/c(n)}$ | Fast, distance-free anomaly scores |

### Where to go next
- Change the `random_state` / `seed` values throughout and watch results shift — that's the
  sensitivity-to-initialization lesson from Section 2, showing up everywhere.
- Swap in your own dataset (`X = your_data`) and re-run any section.
- Ask the tutor: `ask_ai("Compare K-Means and DBSCAN for a dataset with two crescent-moon clusters")`
  (once you've added a real Groq API key).


In [ ]:
print(ask_ai("In two sentences, when should I use DBSCAN instead of K-Means?"))
